In [1]:
from typing import Literal

import torch
from scvi import REGISTRY_KEYS
from scvi.module.base import (
    BaseModuleClass,
    LossOutput,
    auto_move_data,
)
from torch.distributions import NegativeBinomial, Normal
from torch.distributions import kl_divergence as kl
import scvi

In [2]:
adata = scvi.data.synthetic_iid()
adata

AnnData object with n_obs × n_vars = 400 × 100
    obs: 'batch', 'labels'
    uns: 'protein_names'
    obsm: 'accessibility', 'protein_expression'

In [3]:
class MyNN(torch.nn.Module):
    def __init__(self, n_input: int, n_output: int, link_var: Literal["exp", "none", "softmax"]):

        super().__init__()
        self.neural_net = torch.nn.Sequential(
            torch.nn.Linear(n_input, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, n_output),
        )
        
        self.transfromation = None
        if link_var == "exp":
            self.transfromation = torch.exp
        elif link_var == "softmax":
            self.transfromation = torch.nn.Softmax(dim=-1)

    def forward(self, x: torch.Tensor):
        output = self.neural_net(x)    # dakle tu se poziva ne neural net koji je upravo definiro gore sa troch.nn.Sequential, i on ce vratiti output koji je linearna transformacija ulaza, a onda se na taj output primjenjuje transformacija koja je odabrana u konstruktoru klase (exp, softmax ili none)
        if self.transfromation:
            output = self.transfromation(output)
        return output

        

In [4]:
# create the module and observe the architecture. scvi-tools contains many modules to automatically handle complex covariates!
my_neural_net = MyNN(100, 10, "softmax")
my_neural_net

MyNN(
  (neural_net): Sequential(
    (0): Linear(in_features=100, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=10, bias=True)
  )
  (transfromation): Softmax(dim=-1)
)

Building a Module with Vanilla Pytorch

In [5]:
class VAE(BaseModuleClass):
    def __init__(
            self,
            n_input: int,
            n_latent: int
    ):
        super().__init__()
        
        # define decoder as neural network
        self.decoder = MyNN(n_latent, n_input, "softmax") # output means of each gene needs to be non-negative
        self.log_theta = torch.nn.Parameter(torch.randn(n_input)) # this is variance essentially of the output, so for each gene/protein
        #if we predicts something, it wont be optimized necceseraly, so I need to specify that theta is a parameter that needs to be optimized, so I use torch.nn.Parameter, and I initialize it with random values from a normal distribution with mean 0 and standard deviation 1, and the shape of this parameter is (n_input,), which means that we have one variance for each gene/protein in the input data
        # what I DON'T like is that this Theta does not emerge from the Neural Nets themselves, it is just a random number that is optimized

        #defining encoder
        self.mean_encoder = MyNN(n_input, n_latent, "none") # mean of latent var is ok if it is NEGATIVE, it just needs to make sense to the computer
        self.var_encoder = MyNN(n_input, n_latent, "exp") # we want variance to be positive, so we use exp as link function

    def _get_inference_input(self, tensors):
        x = tensors[REGISTRY_KEYS.X_KEY]
        input_dict = dict(x=x)
        return input_dict

    @auto_move_data
    def inference(self, tensors):
        x = tensors[REGISTRY_KEYS.X_KEY]
        x_ = torch.log(x + 1) #logp1 essentially
        qz_m = self.mean_encoder(x_)
        qz_v = self.var_encoder(x_)
        #reparametrization trick
        z = Normal(qz_m, torch.sqrt(qz_v)).rsample() 

        outputs = dict(qz_m=qz_m, qz_v=qz_v, z=z)
        return outputs
    
    def _get_generative_input(self, tensors, inference_outputs):
        z = inference_outputs["z"]
        x = tensors[REGISTRY_KEYS.X_KEY]
        library_size = torch.sum(x, dim=1, keepdim=True) # library size is sum of counts for each cell, so we sum across genes/proteins for each cell

        input_dict = {
            "z": z,
            "library_size": library_size
        }

    @auto_move_data
    def generative(self, z, library_size):
        
        # produces normalized MEAN of the negative binomial - it doesnt really calculate it in a structured way, it just a softmax output of NN
        px_scale = self.decoder(z) 
        # we need to multiply by library size to get the mean of the negative binomial, because the mean of the negative binomial is the product of the normalized mean and the library size
        px_rate = px_scale * library_size
        # theta is the variance of the negative binomial, and it is a parameter that is optimized during training, it is not a function of the input data, it is just a random number that is optimized during training
        theta = torch.exp(self.log_theta) 

        '''
        Shvati da je px_scale jednsotavno array upravo generairan decoderom, možeš ju množit, radit što hoćeš
        Upravo na isti način generiramo i thetu, prije smo definirali log_theta kao nešto što proizvodi array brojki.
        
        Dakle ova f-ja generative() i inferance() samo vrše računanje s funkcijama definiranim u init() - bitno je da skužiš jednostavnost koncepcije
        
        KOliko shvaćam mi ništa nismo logaritmirali, ali koristimo to u eskponentu pa se očekuje da je to vrijednost kojom potenciramo log thateta ako je ciljani rezultat potencijajcije theta
        A ona priča o pozitivnosti je vrlo jednsotavna, e^x > 0 za svaki x element R
        '''

        return dict(px_scale=px_scale, px_rate=px_rate, theta=theta) # px stands for "predicted x"
    
    def loss(self, tensors, inference_outputs, generative_outputs):
        x = tensors[REGISTRY_KEYS.X_KEY]
        qz_m = inference_outputs["qz_m"]
        qz_v = inference_outputs["qz_v"]
        px_rate = generative_outputs["px_rate"]
        theta = generative_outputs["theta"]

        
        # Calculate likelihood loss
        nb_logits = (px_rate + 1e-8).log() - (theta + 1e-8).log() # log of mean minus log of variance, this is the logit parameter of the negative binomial distribution
        log_likelihood = NegativeBinomial(total_count=theta, logits=nb_logits).log_prob(x).sum(dim=-1) # -1 je jer je to column za GENE, tj sumira preko svih gena #nisam bas skužio zašto je total counts = theta, ali uprincipu to je dipersion
        #ovdje smo izračunali sumu svih log_likelihooda, a likelihhodi odgovaraju na pitanje; koja je šansa da izvučemo x iz predviđene ditribucije

        #calculate KL Regularization
        prior_dist = Normal(torch.zeros_like(qz_m), torch.ones_like(qz_v)) # prior distribution is standard normal
        post_dist = Normal(qz_m, torch.sqrt(qz_v)) # variational distribution is normal with mean and variance predicted by encoder
        kl_divergence = kl(post_dist, prior_dist).sum(dim=-1)

        elbo = log_likelihood - kl_divergence
        loss = torch.mean(-elbo) # we want to maximize elbo
        '''
        we obviously take MEAN across all cells, PAZI elbo NIJE jedan broj nego matrica n_cells x 1, 
        1 jer se za gene već sumiralo vrijednossti posteriora i KL preko latents dakle imali smo log_likelihood(n_cells x n_genes).sum
        te KL(n_cells x n_latent).sum
        Tad smo računali SUMU jer ...??
        
        '''
        return LossOutput(loss=loss, recon_loss=-torch.mean(log_likelihood), kl_loss=torch.mean(kl_divergence))

    


In [7]:
VAE(200, 10)

VAE(
  (decoder): MyNN(
    (neural_net): Sequential(
      (0): Linear(in_features=10, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=200, bias=True)
    )
    (transfromation): Softmax(dim=-1)
  )
  (mean_encoder): MyNN(
    (neural_net): Sequential(
      (0): Linear(in_features=200, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=10, bias=True)
    )
  )
  (var_encoder): MyNN(
    (neural_net): Sequential(
      (0): Linear(in_features=200, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=10, bias=True)
    )
  )
)

Wrapping the module in a Model and Training

In [51]:
from typing import Optional, Sequence

import numpy as np
import scvi
import torch
from anndata import AnnData
from scvi import REGISTRY_KEYS
from scvi.data import AnnDataManager
from scvi.data.fields import (
    CategoricalJointObsField,
    CategoricalObsField,
    LayerField,
    NumericalJointObsField,
    NumericalObsField,
)
from scvi.model.base import BaseModelClass, UnsupervisedTrainingMixin, VAEMixin
from scvi.module import VAE

In [56]:
class MyModel(VAEMixin, UnsupervisedTrainingMixin, BaseModelClass):
    def __init__(
            self,
            adata: AnnData,
            #n_input: int, #ne definiriaš fiksan n_input jer input podatci mogu varrirat u dimneijama,  čekaj čeak ali što ako želimo da model bude više manje isti i reproducibilan?
            n_latent: int,
            **model_kwargs
    ):
        super().__init__(adata)

        self.module = VAE(
            n_input = self.summary_stats["n_vars"], # ovo je broj gena/proteina, tj broj kolona u input podatcima
            n_batch = self.summary_stats["n_batch"], # ovo je broj batch-eva, tj broj različitih skupina podataka koje imamo, npr različiti eksperimenti, različiti datumi, različiti laboratoriji itd
            n_latent = n_latent,
            **model_kwargs
        )

        self._model_summary_string = f"My VAE model with {n_latent} latent dimensions and {self.summary_stats.n_batch} batches"

        self.init_params_ = self._get_init_params(locals()) # ča su locals? - riječnik svih varijabli dosad definirianih, bit je da sam mogo pojedniačne var napista, ali ideaj je da ako slučajno u budnućnosti ja dodam još neku varijablu u konstruktor, ona će automatski biti spremljena u init_params_ i neće se zaboraviti, a to je važno za reproducibilnost modela, jer ako želimo reproducirati model, trebamo znati sve parametre koji su korišteni za njegovu inicijalizaciju, a ne samo one koje smo ručno napisali u init_params_

    @classmethod
    def setup_anndata(
        cls,
        adata: AnnData,
        batch_key: Optional[str] = None,
        layer: Optional[str] = None,
        **kwargs,
    ) -> Optional[AnnData]:
        """
        This method registers the fields of the AnnData object to be used in the model. It is called when the model is initialized with an AnnData object. It is also called when the model is loaded from a checkpoint, so it needs to be able to handle both cases.

        Parameters
        ----------
        adata
            The AnnData object to register.
        batch_key
            The key in `adata.obs` that corresponds to batch information. If `None`, no batch information will be registered.
        layer
            The key in `adata.layers` that corresponds to the input data. If `None`, `adata.X` will be used as input data.

        Returns
        -------
        Optional[AnnData]
            The AnnData object with registered fields. If `None`, the original AnnData object will be used.
        """
        setup_method_args = cls._get_setup_method_args(**locals())
        anndata_fields = [
            LayerField(REGISTRY_KEYS.X_KEY, layer, is_count_data=True),
            CategoricalObsField(REGISTRY_KEYS.BATCH_KEY, batch_key),
            # Dummy fields required for VAE class.
            CategoricalObsField(REGISTRY_KEYS.LABELS_KEY, None),
            NumericalObsField(REGISTRY_KEYS.SIZE_FACTOR_KEY, None, required=False),
            CategoricalJointObsField(REGISTRY_KEYS.CAT_COVS_KEY, None),
            NumericalJointObsField(REGISTRY_KEYS.CONT_COVS_KEY, None),
        ]

        '''
            Ovu dummy fields su tu samo da zadovolje strukturu koju scvi-tools očekuje, ali obzirom da nemam JointObs 
            tj. matrice koje opisuju covariates to je jednsotavno prazno

            '''
    
        adata_manager = AnnDataManager(
            fields=anndata_fields,
            setup_method_args=setup_method_args,    
        )
        
        adata_manager.register_fields(adata, **kwargs)
        cls.register_manager(adata_manager)


pazi ništ aod ovog nije neštoo jako komplkeso, samo se pozva na puno pozadinskih funkcija koje ja nisam moro definirat, samo moras znat kaj one delaju i kaj outpuaju, ali u prinicpu cijela ova classmethod je samo par funkcija psojene u automatsk niz

In [53]:
adata = scvi.data.synthetic_iid()
adata

AnnData object with n_obs × n_vars = 400 × 100
    obs: 'batch', 'labels'
    uns: 'protein_names'
    obsm: 'accessibility', 'protein_expression'

In [ ]:
MyModel.setup_anndata(adata, batch_key="batch")    
print(f"adata UUID (assigned by setup_anndata): {adata.uns['_scvi_uuid']}")
print(f"AnnDataManager: {MyModel._setup_adata_manager_store[adata.uns['_scvi_uuid']]}")
model = MyModel(adata, n_latent=20)
model

adata UUID (assigned by setup_anndata): da4ae282-6e88-48f7-b3dc-3433c07bc81d
AnnDataManager: <scvi.data._manager.AnnDataManager object at 0x7fff4f984510>


My VAE model with 10 latent dimensions and 2 batches
Training status: Not Trained

In [30]:
model._validate_anndata(adata)

AnnData object with n_obs × n_vars = 400 × 100
    obs: 'batch', 'labels', '_scvi_labels', '_scvi_batch'
    uns: 'protein_names', '_scvi_uuid', '_scvi_manager_uuid'
    obsm: 'accessibility', 'protein_expression'

In [58]:
model.train(max_epochs=5, train_size=0.9)

/g/stegle/spiljak/programs/miniforge3/envs/scvi-env2/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /g/stegle/spiljak/programs/miniforge3/envs/scvi-env2 ...
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/g/stegle/spiljak/programs/miniforge3/envs/scvi-env2/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /g/stegle/sp

Epoch 5/5: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 45.28it/s, v_num=1, train_loss=317]

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 5/5: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 44.20it/s, v_num=1, train_loss=317]


In [ ]:
model.get_latent_representation() # this is z_m 

array([[ 0.53300476, -0.55241096,  0.0930565 , ...,  0.7866237 ,
        -0.02485396, -0.29965207],
       [ 0.11988743,  0.1400927 ,  0.5346062 , ...,  0.24744987,
         0.14803839, -0.3847268 ],
       [ 0.3361353 , -0.41784063,  0.2699482 , ...,  0.49073452,
         0.5510265 , -0.17968485],
       ...,
       [ 0.24815156,  0.137486  ,  0.33724508, ...,  0.3804522 ,
         0.21336794, -0.37652493],
       [ 0.31783882,  0.10930268,  0.4071393 , ...,  0.2319858 ,
         0.3701585 , -0.27155125],
       [ 0.9160035 , -0.4095826 ,  0.2603891 , ...,  0.22957684,
        -0.3318119 , -0.0606642 ]], shape=(400, 10), dtype=float32)

In [37]:
model.save("saved_model/", save_anndata=True)
model = MyModel.load("saved_model/")

INFO     File saved_model/model.pt already downloaded                                                              


/g/stegle/spiljak/programs/miniforge3/envs/scvi-env2/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /g/stegle/spiljak/programs/miniforge3/envs/scvi-env2 ...
